# 08 · Access: baseline-лестница и Deep Sets (GPU)

**Вычисления:** Kaggle **GPU (NVIDIA)**. FULL без GPU останавливается — CPU-суррогат запрещён контрактом.

## Цель

`target_access_corroboration_24_48h_proxy`: в сутки **D+2** на объекте наблюдались тревожные сигналы
не менее чем от **двух разных access-типов**. Признаки — до конца суток D, сутки D+1 — запас на реакцию.
**Это не факт несанкционированного доступа**, а weak-label proxy уровня E2.

## Модели (на одних и тех же validation-строках)

| код | модель |
|---|---|
| A0 | глобальная prevalence (train) |
| A1 | prevalence по составу access-типов объекта (train, глобальный fallback) |
| A_rec | recurrence: `1/(1+d_days_since_access_corroboration)` |
| A2 | детерминированное multi-signal правило |
| A3 | регуляризованная логистическая регрессия на признаках объекта (без календаря) |
| A4 | Deep Sets на наборе channel-элементов объекта |

Deep Sets: общий `phi` для каждого канала (16 числовых признаков + эмбеддинг `тип_датчика`,
неизвестный тип — зарезервированный индекс 0), masked mean+max pooling (паддинг в pooling не участвует),
`rho` — бинарная голова; в `rho` дополнительно подаётся recurrence объекта (как у A3).

## Протокол

- Фолды: `fold_2023` (train — окна цели до 2023-01-01, calibration 2023 H1, validation 2023 H2) и `fold_2024`
  (до 2024-01-01, 2024 H1, 2024 H2). 2021 не входит в train. **2025 H2 и 2026 не используются**: всё с окном цели
  после 2025-01-01 отбрасывается сразу после проверки контракта.
- Train: отрицательные прорежены 20:1 с inverse-probability весами. Validation не прореживается.
- Скейлеры, словарь типов, составы, Platt-калибровка, порог и ранняя остановка Deep Sets используют только train и calibration H1.
- Календарь (`d_month`, `d_weekday`) не используется.
- Метрики: PR-AUC и lift над prevalence, Brier, ECE, top-K тревог объектов в сутки, эпизоды (семантика 07)
  при бюджетах 5/10/25/50 и cooldown 24/72/168 ч, ложные тревоги на 1000 object-days, clean-history
  (recurrence ≥ 30 суток или null) и seen/unseen object срезы, парные 95% CI бутстрэпом блоками календарных недель.
- Отбор симметричный: нейросеть выбирается, только если парный CI по основной метрике выше нуля, episode precision
  не хуже лучшей простой модели более чем на 1 п.п. и ложные тревоги не растут. При ничьей — простая модель.

## Важно про бюджет 50 объектов/сутки

Контракт задаёт основной бюджет **50 тревог объектов в сутки**. Объектов в панели порядка десятков, поэтому такой бюджет
может покрывать почти все активные объекты — тогда эпизодный recall одинаков у всех моделей. Ноутбук это измеряет
(`budget_saturation`) и при насыщении ≥ 90% ставит `decision_status = not_evaluable_primary_budget_saturated`
вместо «выбора» константы. Чувствительность при бюджете 5 выводится рядом.

## Как запустить

1. **Add Input** → private Dataset с **FULL-выходом notebook 07** (`missing_targets_manifest_v1.json` ищется автоматически).
   Smoke-бандл 07 не подходит: в нём нет фолдов 2023/2024.
2. **Settings → Accelerator → GPU.**
3. Сначала `run_mode = "smoke"` (треть объектов, 3 эпохи, 50 повторов бутстрэпа; помечается `non_comparable_smoke`), затем `"full"`.
4. **Run All.**

In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО МЕНЯТЬ
TASK = "access"
CONFIG = {
    "run_mode": "full",           # "smoke" или "full"; smoke несравним
    "alert_budget_per_day": 50,   # основной бюджет по контракту; при насыщении см. вывод
    "random_seed": 20260922,
}
print("Notebook 08 | задача:", TASK, "| режим:", CONFIG["run_mode"])

In [ ]:
# Проверка среды и GPU preflight
import importlib.util, subprocess, sys
for module, spec in {"polars": "polars>=1.0,<2", "sklearn": "scikit-learn>=1.4,<2", "torch": "torch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
else:
    print("GPU не найден.", "FULL будет остановлен (нужен Accelerator = GPU)." if CONFIG["run_mode"] == "full"
          else "SMOKE выполнится на CPU и будет помечен как non_comparable_smoke.")

## Встроенный код

Первая ячейка — выдержка из runtime 07 без изменения логики (загрузка бандла, temporal split, калибровка, порог, эпизодные метрики). Вторая — модели и отбор 08/09. Третья — самопроверки.

In [ ]:
"""Выдержка из общего runtime notebook 07 (missing-targets-1.1), скопирована без изменений логики.

Используется 08/09 для загрузки бандла, temporal split, калибровки, порога и эпизодных метрик,
чтобы определения совпадали с 07 и 10–12.
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta
import hashlib
import json
import math
import os
from pathlib import Path
import platform
from typing import Any, Sequence

import numpy as np


SOURCE_SCHEMA_VERSION = "2.0"
DERIVED_SCHEMA_VERSION = "missing-targets-1.1"
ACCESS_TARGET = "target_access_corroboration_24_48h_proxy"
FIRE_TARGET = "target_fire_corroboration_24_48h_proxy"
DROPOUT_TARGET = "target_channel_dropout_24_48h_proxy"
GENERIC_ALARM_TARGET = "target_any_alarm_observed_Dplus2_proxy"

ACCESS_TYPES = {
    "КД Дверь",
    "КД АВ",
    "КД Люк",
    "9-секционный люк",
    "Датчик движения",
    "Состояние охраны",
    "Стекло",
}
FIRE_TYPES = {
    "Датчик дыма",
    "Газовый датчик",
    "Датчик температуры",
    "Тепловой датчик",
    "Ручной извещатель",
}

OBJECT_FEATURES = [
    "d_channel_count",
    "d_sensor_type_count",
    "d_event_count_sum_24h",
    "d_alarm_count_sum_24h",
    "d_alarm_share_mean_24h",
    "d_failure_state_count_sum_24h",
    "d_numeric_mean_mean_24h",
    "d_numeric_min_24h",
    "d_numeric_max_24h",
    "d_numeric_std_mean_24h",
    "d_state_n_unique_max_24h",
    "d_gap_days_mean",
    "d_access_channel_count",
    "d_access_alarm_type_count",
    "d_fire_channel_count",
    "d_fire_alarm_type_count",
    "d_ventilation_alarm_count_24h",
    "d_flood_channel_count",
    "d_flood_alarm_count_24h",
    "d_pump_channel_count",
    "d_pump_alarm_count_24h",
    "d_power_alarm_count_24h",
    "d_flood_numeric_mean_24h",
    "d_pump_numeric_mean_24h",
    "d_days_since_access_corroboration",
    "d_days_since_fire_corroboration",
]

ELEMENT_NUMERIC_FEATURES = [
    "d_event_count_24h",
    "d_alarm_count_24h",
    "d_alarm_share_24h",
    "d_failure_state_event_count_24h",
    "d_value_numeric_mean_24h",
    "d_value_numeric_min_24h",
    "d_value_numeric_max_24h",
    "d_value_numeric_std_24h",
    "d_value_numeric_last",
    "d_state_n_unique_24h",
    "d_gap_days_since_previous",
    "d_event_count_previous_24h",
    "d_alarm_count_previous_24h",
    "d_alarm_share_previous_24h",
    "d_value_numeric_previous",
    "d_catalogue_match",
]

TARGET_REGISTRY = {
    ACCESS_TARGET: {
        "evidence_level": "E2",
        "score_kind": "calibrated_corroboration_proxy_probability",
        "entity": "object-day",
        "meaning_ru": (
            "В сутки D+2 на объекте наблюдались тревожные сигналы не менее чем "
            "от двух разных типов access-каналов. Это не факт несанкционированного доступа."
        ),
    },
    FIRE_TARGET: {
        "evidence_level": "E2",
        "score_kind": "calibrated_corroboration_proxy_probability",
        "entity": "object-day",
        "meaning_ru": (
            "В сутки D+2 на объекте наблюдались тревожные сигналы не менее чем "
            "от двух разных типов fire-каналов. Это не подтверждённый пожар."
        ),
    },
}


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _discover_one(name: str, explicit: str | Path | None = None) -> Path:
    candidates: list[Path] = []
    if explicit:
        root = Path(explicit)
        candidates.extend([root / name, *root.rglob(name)])
    env_root = os.environ.get("LDT_KAGGLE_DATA_DIR")
    if env_root:
        root = Path(env_root)
        candidates.extend([root / name, *root.rglob(name)])
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(kaggle_root.rglob(name))
    unique = sorted({path.resolve() for path in candidates if path.is_file()})
    if len(unique) != 1:
        raise RuntimeError(
            f"Ожидался ровно один {name}; найдено: {len(unique)}. "
            "Прикрепите только нужный private Kaggle Dataset."
        )
    return unique[0]


def load_missing_target_bundle(
    explicit: str | Path | None = None,
) -> tuple[Any, Any, dict[str, Any], Path]:
    import polars as pl

    manifest_path = _discover_one("missing_targets_manifest_v1.json", explicit)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != DERIVED_SCHEMA_VERSION:
        raise RuntimeError("Несовместимая версия missing-target panel")
    if manifest.get("raw_identifiers_included") is not False:
        raise RuntimeError("Нарушен de-identification contract")
    paths = {
        key: manifest_path.parent / filename for key, filename in manifest["files"].items()
    }
    for key in ("object_day", "channel_element", "registry"):
        if sha256_file(paths[key]) != manifest["sha256"][key]:
            raise RuntimeError(f"Не совпала SHA-256: {key}")
    object_day = pl.read_parquet(paths["object_day"])
    channel = pl.read_parquet(paths["channel_element"])
    if object_day.height != manifest["rows"]["object_day"]:
        raise RuntimeError("Не совпало число object-day строк")
    if channel.height != manifest["rows"]["channel_element"]:
        raise RuntimeError("Не совпало число channel-element строк")
    return object_day, channel, manifest, manifest_path


@dataclass(frozen=True)
class RollingFold:
    name: str
    train_end: date
    calibration_start: date
    calibration_end: date
    validation_start: date
    validation_end: date


ROLLING_FOLDS = [
    RollingFold(
        "fold_2023",
        date(2023, 1, 1),
        date(2023, 1, 1),
        date(2023, 7, 1),
        date(2023, 7, 1),
        date(2024, 1, 1),
    ),
    RollingFold(
        "fold_2024",
        date(2024, 1, 1),
        date(2024, 1, 1),
        date(2024, 7, 1),
        date(2024, 7, 1),
        date(2025, 1, 1),
    ),
]


def rolling_split(
    frame: Any,
    target: str,
    fold: RollingFold,
    start_column: str = "d_target_start_date",
    end_column: str = "d_target_end_date_exclusive",
) -> dict[str, Any]:
    """Split by fully observed target windows; 2021 is never training data."""
    import polars as pl

    labelled = frame.filter(pl.col(target).is_not_null())
    start = pl.col(start_column)
    end = pl.col(end_column)
    cutoff_year = pl.col("d_cutoff_date").dt.year()
    train = labelled.filter((end <= fold.train_end) & (cutoff_year != 2021))
    calibration = labelled.filter(
        (start >= fold.calibration_start) & (end <= fold.calibration_end)
    )
    validation = labelled.filter(
        (start >= fold.validation_start) & (end <= fold.validation_end)
    )
    if min(train.height, calibration.height, validation.height) == 0:
        raise RuntimeError(f"Пустой temporal split: {fold.name}")
    return {"train": train, "calibration": calibration, "validation": validation}


def sample_training_rows(
    frame: Any, target: str, ratio: int, seed: int
) -> tuple[Any, np.ndarray]:
    import polars as pl

    positives = frame.filter(pl.col(target) == 1)
    negatives = frame.filter(pl.col(target) == 0)
    if positives.height == 0:
        raise RuntimeError("В train нет positives")
    keep = min(negatives.height, max(positives.height * ratio, 1))
    sampled_negatives = negatives.sample(n=keep, seed=seed, shuffle=True)
    sampled = pl.concat([positives, sampled_negatives]).sample(
        fraction=1.0, seed=seed, shuffle=True
    )
    negative_probability = keep / max(negatives.height, 1)
    y = sampled[target].to_numpy()
    weights = np.where(y == 1, 1.0, 1.0 / max(negative_probability, 1e-12))
    return sampled, weights.astype(np.float64)


def _finite_probability(values: Sequence[float]) -> np.ndarray:
    result = np.asarray(values, dtype=np.float64)
    return np.clip(np.nan_to_num(result, nan=0.0, posinf=1.0, neginf=0.0), 0, 1)


def fit_platt(raw_probability: Sequence[float], y: Sequence[int]) -> Any:
    from sklearn.linear_model import LogisticRegression

    p = np.clip(np.asarray(raw_probability, dtype=np.float64), 1e-6, 1 - 1e-6)
    labels = np.asarray(y, dtype=np.int8)
    if np.unique(labels).size < 2:
        return {"kind": "identity"}
    logits = np.log(p / (1 - p)).reshape(-1, 1)
    model = LogisticRegression(random_state=0, max_iter=500)
    model.fit(logits, labels)
    return {"kind": "platt", "model": model}


def apply_platt(spec: Any, raw_probability: Sequence[float]) -> np.ndarray:
    p = np.clip(np.asarray(raw_probability, dtype=np.float64), 1e-6, 1 - 1e-6)
    if spec["kind"] == "identity":
        return p
    logits = np.log(p / (1 - p)).reshape(-1, 1)
    return _finite_probability(spec["model"].predict_proba(logits)[:, 1])


def expected_calibration_error(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:
    edges = np.linspace(0, 1, bins + 1)
    total = len(y)
    error = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (p >= left) & (p < right if right < 1 else p <= right)
        if mask.any():
            error += mask.mean() * abs(float(y[mask].mean()) - float(p[mask].mean()))
    return float(error)


def threshold_for_precision(y: np.ndarray, p: np.ndarray, minimum: float) -> float:
    from sklearn.metrics import precision_recall_curve

    if np.unique(y).size < 2:
        return 1.0
    precision, recall, thresholds = precision_recall_curve(y, p)
    candidates = [
        (float(recall[index]), float(thresholds[index]))
        for index in range(len(thresholds))
        if precision[index] >= minimum
    ]
    return max(candidates, default=(0.0, 1.0))[1]


def _top_budget_mask(dates: np.ndarray, probability: np.ndarray, budget: int) -> np.ndarray:
    mask = np.zeros(len(probability), dtype=bool)
    for current in np.unique(dates):
        indices = np.flatnonzero(dates == current)
        selected = indices[np.argsort(-probability[indices], kind="stable")[:budget]]
        mask[selected] = True
    return mask


def episode_metrics(
    frame: Any,
    target: str,
    probability: Sequence[float],
    budget: int,
    cooldown_hours: int,
    threshold: float | None = None,
    entity_column: str = "d_object_key",
) -> dict[str, Any]:
    import pandas as pd

    columns = [entity_column, "d_cutoff_date", "d_target_start_date", target]
    data = pd.DataFrame(frame.select(columns).to_dict(as_series=False))
    data["probability"] = _finite_probability(probability)
    data["d_cutoff_date"] = pd.to_datetime(data["d_cutoff_date"])
    data["d_target_start_date"] = pd.to_datetime(data["d_target_start_date"])
    top = _top_budget_mask(
        data["d_cutoff_date"].dt.date.to_numpy(), data["probability"].to_numpy(), budget
    )
    if threshold is not None:
        top &= data["probability"].to_numpy() >= threshold
    alerts = data.loc[top].sort_values(["d_cutoff_date", "probability"], ascending=[True, False])
    cooldown = timedelta(hours=int(cooldown_hours))
    accepted: list[int] = []
    last: dict[str, Any] = {}
    for index, row in alerts.iterrows():
        entity = row[entity_column]
        if entity not in last or row["d_cutoff_date"] - last[entity] >= cooldown:
            accepted.append(index)
            last[entity] = row["d_cutoff_date"]
    accepted_frame = data.loc[accepted]
    events = data.loc[data[target] == 1, [entity_column, "d_target_start_date"]].drop_duplicates()
    event_set = set(map(tuple, events.to_numpy()))
    matched: set[tuple[Any, Any]] = set()
    for _, row in accepted_frame.iterrows():
        key = (row[entity_column], row["d_target_start_date"])
        if key in event_set:
            matched.add(key)
    n_alerts = len(accepted_frame)
    n_events = len(event_set)
    return {
        "alert_episodes": n_alerts,
        "proxy_episodes": n_events,
        "matched_episodes": len(matched),
        "precision": float(len(matched) / n_alerts) if n_alerts else 0.0,
        "recall": float(len(matched) / n_events) if n_events else 0.0,
        "alert_budget_per_day": budget,
        "cooldown_hours": cooldown_hours,
    }


def _robust_fit(frame: Any, features: Sequence[str]) -> dict[str, np.ndarray]:
    values = frame.select(features).to_numpy().astype(np.float64)
    median = np.nanmedian(values, axis=0)
    q25 = np.nanquantile(values, 0.25, axis=0)
    q75 = np.nanquantile(values, 0.75, axis=0)
    scale = q75 - q25
    scale[~np.isfinite(scale) | (scale < 1e-6)] = 1.0
    median[~np.isfinite(median)] = 0.0
    return {"median": median, "scale": scale}


def _robust_apply(frame: Any, features: Sequence[str], spec: dict[str, np.ndarray]) -> np.ndarray:
    values = frame.select(features).to_numpy().astype(np.float64)
    values = np.where(np.isfinite(values), values, spec["median"])
    return np.clip((values - spec["median"]) / spec["scale"], -20, 20).astype(
        np.float32
    )


In [ ]:
"""Runtime notebooks 08 (access) и 09 (fire): baseline-лестница, Deep Sets, отбор.

Опирается на выдержку runtime 07 (ячейка выше): rolling_split, sample_training_rows,
fit_platt/apply_platt, threshold_for_precision, episode_metrics, _robust_fit/_robust_apply.
Схема 07 не меняется: при отсутствии поля — остановка с понятной ошибкой.
"""

from __future__ import annotations


import copy
import gc
import json
import math
import os
import platform
import re
import sys
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import numpy as np

LOAD_END_EXCLUSIVE = date(2025, 1, 1)
RAW_ID_COLUMNS = {"ид_канала_данных", "ид_объект", "ид_события"}

TASKS = {
    "access": {
        "target": ACCESS_TARGET,
        "family_types": ACCESS_TYPES,
        "days_since": "d_days_since_access_corroboration",
        "alarm_type_count": "d_access_alarm_type_count",
        "channel_count": "d_access_channel_count",
        "prefix": "A",
        "outputs": ("results_access_missing_target.json", "predictions_access_missing_target.parquet",
                    "summary_access_missing_target_ru.md"),
        "artifact_stem": "model_access_selected",
        "title_ru": "Access: корроборация тревог ≥2 access-типов в D+2",
    },
    "fire": {
        "target": FIRE_TARGET,
        "family_types": FIRE_TYPES,
        "days_since": "d_days_since_fire_corroboration",
        "alarm_type_count": "d_fire_alarm_type_count",
        "channel_count": "d_fire_channel_count",
        "prefix": "F",
        "outputs": ("results_fire_missing_target.json", "predictions_fire_missing_target.parquet",
                    "summary_fire_missing_target_ru.md"),
        "artifact_stem": "model_fire_selected",
        "title_ru": "Fire: корроборация тревог ≥2 fire-типов в D+2",
    },
}
SIMPLE_ORDER = ["prevalence", "composition_prevalence", "recurrence", "rule", "logistic"]
MODEL_LABELS = {
    "access": {"prevalence": "A0", "composition_prevalence": "A1", "recurrence": "A_rec", "rule": "A2",
               "logistic": "A3", "deepsets": "A4"},
    "fire": {"prevalence": "F0", "composition_prevalence": "F_comp", "recurrence": "F_rec", "rule": "F1",
             "logistic": "F2", "deepsets": "F3", "deepsets_transfer": "F4", "deepsets_smooth_ap": "F5"},
}
NEURAL = {"deepsets", "deepsets_transfer", "deepsets_smooth_ap"}


class ContractError(RuntimeError):
    """Нарушен контракт входных данных 07."""


class DeviceError(RuntimeError):
    """FULL требует NVIDIA GPU."""


def log(*parts: Any) -> None:
    print(time.strftime("%H:%M:%S"), *parts, flush=True)


def _clean(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_clean(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (date,)):
        return value.isoformat()
    if isinstance(value, Path):
        return value.name
    return value


def device_info() -> dict[str, Any]:
    info: dict[str, Any] = {"python": platform.python_version(), "platform": platform.platform(),
                            "cpu_count": os.cpu_count()}
    for name in ("numpy", "polars", "sklearn", "torch"):
        try:
            info[name] = __import__(name).__version__
        except Exception:
            info[name] = None
    try:
        import torch

        info["cuda_available"] = bool(torch.cuda.is_available())
        info["device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    except Exception:
        info["cuda_available"], info["device"] = False, "CPU"
    return info


# ---------------------------------------------------------------------------
# 1. Контракт 07 и загрузка
# ---------------------------------------------------------------------------
def load_bundle_checked(task: str, config: dict[str, Any]) -> tuple[Any, Any, dict[str, Any], dict[str, Any]]:
    import polars as pl

    spec = TASKS[task]
    try:
        object_day, channel, manifest, _ = load_missing_target_bundle()
    except RuntimeError as error:
        raise ContractError(f"Бандл 07 не прошёл проверку: {error}") from error
    checks: dict[str, Any] = {"schema_version": manifest.get("schema_version"), "sha256_match": True,
                              "row_counts_match": True}
    if manifest.get("contains_2026_rows") is not False:
        raise ContractError("Манифест 07 не подтверждает отсутствие 2026 года")
    if manifest.get("run_mode") != "full":
        raise ContractError(
            "Бандл 07 собран в режиме smoke: в нём только последние ~120 дней, фолды 2023/2024 "
            "построить нельзя. Для 08/09 (включая их smoke) нужен FULL-бандл 07.")
    for name, frame in (("object_day", object_day), ("channel_element", channel)):
        raw = sorted(RAW_ID_COLUMNS & set(frame.columns))
        if raw:
            raise ContractError(f"{name}: найдены сырые идентификаторы {raw}")
        year_max = frame.select(pl.col("d_cutoff_date").dt.year().max()).item()
        if year_max is not None and year_max >= 2026:
            raise ContractError(f"{name}: есть строки 2026 года")
    required_object = set(OBJECT_FEATURES) | {
        spec["target"], "d_object_key", "d_cutoff_date", "d_target_start_date",
        "d_target_end_date_exclusive", spec["days_since"], spec["alarm_type_count"], spec["channel_count"]}
    if task == "fire":
        required_object.add(GENERIC_ALARM_TARGET)
    required_channel = set(ELEMENT_NUMERIC_FEATURES) | {"d_object_key", "d_cutoff_date", "тип_датчика"}
    for name, frame, required in (("object_day", object_day, required_object),
                                  ("channel_element", channel, required_channel)):
        missing = sorted(required - set(frame.columns))
        if missing:
            raise ContractError(f"{name}: нет обязательных полей контракта 07: {missing}")
    for key in ("date_min", "date_max"):
        actual = str(getattr(object_day["d_cutoff_date"], key.split("_")[1])())
        if str(manifest.get(key))[:10] != actual[:10]:
            raise ContractError(f"{key}: в манифесте {manifest.get(key)}, в object_day {actual}")
    checks.update(date_min=manifest.get("date_min"), date_max=manifest.get("date_max"),
                  rows_object_day_total=object_day.height, rows_channel_total=channel.height,
                  required_columns_present=True, raw_identifiers_absent=True)
    # Ничего с окном цели после 2025-01-01 дальше не используется.
    object_day = object_day.filter(pl.col("d_target_end_date_exclusive") <= pl.lit(LOAD_END_EXCLUSIVE))
    channel = channel.filter(pl.col("d_cutoff_date") < pl.lit(LOAD_END_EXCLUSIVE - timedelta(days=2)))
    if config["run_mode"] == "smoke":
        objects = object_day["d_object_key"].unique().sort()
        keep = objects.filter(objects.hash(seed=20260922) % int(config["smoke_object_share"]) == 0)
        if keep.len() < 3:
            keep = objects.head(max(3, objects.len() // 3))
        object_day = object_day.filter(pl.col("d_object_key").is_in(keep.implode()))
        channel = channel.filter(pl.col("d_object_key").is_in(keep.implode()))
    object_day = object_day.sort(["d_object_key", "d_cutoff_date"])
    checks.update(rows_object_day_used=object_day.height, rows_channel_used=channel.height,
                  max_target_end_used=str(object_day["d_target_end_date_exclusive"].max()))
    return object_day, channel, manifest, checks


def attach_elements(object_day: Any, channel: Any, family_types: set[str]) -> tuple[Any, dict[str, Any]]:
    """Индексирует channel-элементы каждой object-day строки и сигнатуру состава семейства."""
    import polars as pl

    od = object_day.with_row_index("_od")
    joined = channel.join(od.select("d_object_key", "d_cutoff_date", "_od"),
                          on=["d_object_key", "d_cutoff_date"], how="inner").sort("_od")
    counts = joined.group_by("_od").len().sort("_od")
    lengths = np.zeros(od.height, dtype=np.int64)
    lengths[counts["_od"].to_numpy()] = counts["len"].to_numpy()
    offsets = np.r_[0, np.cumsum(lengths)]
    numeric = joined.select([pl.col(c).cast(pl.Float64) for c in ELEMENT_NUMERIC_FEATURES]).to_numpy()
    types = joined["тип_датчика"].fill_null("__MISSING__").to_numpy()
    signature = (
        joined.filter(pl.col("тип_датчика").is_in(sorted(family_types)))
        .group_by("_od").agg(pl.col("тип_датчика").unique().sort().str.join("|").alias("_sig"))
    )
    od = od.join(signature, on="_od", how="left").with_columns(pl.col("_sig").fill_null("∅")).sort("_od")
    elements = {"numeric": numeric, "types": types, "offsets": offsets, "lengths": lengths}
    return od, elements


# ---------------------------------------------------------------------------
# 2. Простые модели
# ---------------------------------------------------------------------------
def recurrence_score(frame: Any, column: str) -> np.ndarray:
    days = frame[column].cast(float).to_numpy().astype(float)
    out = np.zeros_like(days)
    ok = np.isfinite(days)
    out[ok] = 1.0 / (1.0 + np.maximum(days[ok], 0.0))
    return out


def rule_score(frame: Any, spec: dict[str, Any]) -> np.ndarray:
    """Детерминированное multi-signal правило (без подбора)."""
    types = frame[spec["alarm_type_count"]].fill_null(0).cast(float).to_numpy()
    rec = recurrence_score(frame, spec["days_since"])
    corroborated_today = np.minimum(types / 2.0, 1.0)
    single_today = 0.25 * (types >= 1)
    return np.clip(np.maximum.reduce([corroborated_today, single_today, 0.5 * rec]), 0, 1)


def _signed_log(x: np.ndarray) -> np.ndarray:
    return np.sign(x) * np.log1p(np.abs(x))


class LogisticObject:
    def __init__(self, seed: int):
        self.seed = seed

    def _x(self, frame: Any) -> np.ndarray:
        import polars as pl

        values = frame.select([pl.col(c).cast(pl.Float64) for c in OBJECT_FEATURES]).to_numpy()
        return _signed_log(values)

    def fit(self, train: Any, y: np.ndarray, w: np.ndarray) -> "LogisticObject":
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler

        self.model = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                               ("clf", LogisticRegression(C=0.5, max_iter=1000, solver="liblinear",
                                                          random_state=self.seed))])
        self.model.fit(self._x(train), y, clf__sample_weight=w)
        return self

    def raw(self, frame: Any) -> np.ndarray:
        return self.model.predict_proba(self._x(frame))[:, 1]


def composition_table(train: Any, target: str, min_rows: int) -> tuple[dict[str, float], float]:
    import polars as pl

    global_rate = float(train[target].mean())
    table = train.group_by("_sig").agg(pl.len().alias("n"), pl.col(target).mean().alias("rate"))
    mapping = {r["_sig"]: float(r["rate"]) for r in table.iter_rows(named=True) if r["n"] >= min_rows}
    return mapping, global_rate


# ---------------------------------------------------------------------------
# 3. Deep Sets
# ---------------------------------------------------------------------------
def _torch_device(config: dict[str, Any]) -> Any:
    import torch

    if torch.cuda.is_available():
        return torch.device("cuda")
    if config["run_mode"] == "full":
        raise DeviceError("FULL требует Kaggle NVIDIA GPU (Settings → Accelerator → GPU). "
                          "CPU-суррогат для FULL запрещён контрактом.")
    return torch.device("cpu")


class ElementBatcher:
    """Собирает паддинг-батчи наборов channel-элементов; паддинг помечен маской."""

    def __init__(self, elements: dict[str, Any], robust: dict[str, np.ndarray], vocab: dict[str, int],
                 context: np.ndarray, max_set: int):
        numeric = elements["numeric"]
        values = np.where(np.isfinite(numeric), numeric, robust["median"])
        self.x = np.clip((values - robust["median"]) / robust["scale"], -20, 20).astype(np.float32)
        self.t = np.array([vocab.get(v, 0) for v in elements["types"].tolist()], dtype=np.int64)
        self.offsets, self.lengths = elements["offsets"], elements["lengths"]
        self.context = context.astype(np.float32)
        self.max_set = max_set
        self.truncated_sets = int((self.lengths > max_set).sum())

    def batch(self, rows: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        lengths = np.minimum(self.lengths[rows], self.max_set)
        width = max(int(lengths.max()) if len(lengths) else 1, 1)
        x = np.zeros((len(rows), width, self.x.shape[1]), dtype=np.float32)
        t = np.zeros((len(rows), width), dtype=np.int64)
        mask = np.zeros((len(rows), width), dtype=np.float32)
        for i, (row, n) in enumerate(zip(rows.tolist(), lengths.tolist())):
            start = self.offsets[row]
            x[i, :n] = self.x[start:start + n]
            t[i, :n] = self.t[start:start + n]
            mask[i, :n] = 1.0
        return x, t, mask, self.context[rows]


def build_deepsets(n_numeric: int, n_types: int, n_context: int, config: dict[str, Any]) -> Any:
    import torch
    from torch import nn

    hidden, emb = int(config["ds_hidden"]), int(config["ds_type_embedding"])

    class DeepSets(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            # индекс 0 зарезервирован под неизвестный тип (и паддинг, который маскируется)
            self.embedding = nn.Embedding(n_types + 1, emb)
            self.phi = nn.Sequential(nn.Linear(n_numeric + emb, hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU())
            self.rho = nn.Sequential(nn.Linear(2 * hidden + n_context, hidden), nn.ReLU(),
                                     nn.Dropout(float(config["ds_dropout"])), nn.Linear(hidden, 1))

        def forward(self, x: Any, t: Any, mask: Any, context: Any) -> Any:
            h = self.phi(torch.cat([x, self.embedding(t)], dim=-1))
            m = mask.unsqueeze(-1)
            count = m.sum(dim=1).clamp_min(1.0)
            mean = (h * m).sum(dim=1) / count
            maximum = h.masked_fill(m == 0, float("-inf")).max(dim=1).values
            maximum = torch.where(torch.isfinite(maximum), maximum, torch.zeros_like(maximum))
            return self.rho(torch.cat([mean, maximum, context], dim=-1)).squeeze(-1)

    return DeepSets()


def smooth_ap_loss(logits: Any, y: Any, tau: float) -> Any:
    """Smooth-AP (Brown et al., 2020): дифференцируемая аппроксимация AP внутри батча."""
    import torch

    positive = y > 0.5
    if positive.sum() == 0:
        return logits.sum() * 0.0
    s_pos = logits[positive]
    diff = (logits.unsqueeze(0) - s_pos.unsqueeze(1)) / tau          # [P, B]: s_j - s_i
    sig = torch.sigmoid(diff)
    self_mask = torch.zeros_like(sig, dtype=torch.bool)
    pos_idx = torch.nonzero(positive).squeeze(1)
    self_mask[torch.arange(len(pos_idx)), pos_idx] = True
    sig = sig.masked_fill(self_mask, 0.0)
    rank_all = 1.0 + sig.sum(dim=1)
    rank_pos = 1.0 + (sig * positive.float().unsqueeze(0)).sum(dim=1)
    return 1.0 - (rank_pos / rank_all).mean()


def _predict_ds(model: Any, batcher: ElementBatcher, rows: np.ndarray, device: Any, batch_size: int) -> np.ndarray:
    import torch

    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            x, t, m, c = batcher.batch(rows[start:start + batch_size])
            logits = model(torch.from_numpy(x).to(device), torch.from_numpy(t).to(device),
                           torch.from_numpy(m).to(device), torch.from_numpy(c).to(device))
            out.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def train_deepsets(model: Any, batcher: ElementBatcher, train_rows: np.ndarray, y_train: np.ndarray,
                   w_train: np.ndarray, cal_rows: np.ndarray, y_cal: np.ndarray, config: dict[str, Any],
                   device: Any, loss_kind: str = "bce", lr: float | None = None, epochs: int | None = None,
                   seed: int = 0) -> tuple[Any, dict[str, Any]]:
    """Обучение с ранней остановкой по PR-AUC на calibration H1 (validation не используется)."""
    import torch
    from sklearn.metrics import average_precision_score

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    model = model.to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr or float(config["ds_lr"]), weight_decay=1e-4)
    batch_size = int(config["ds_batch_size"])
    epochs = epochs or int(config["ds_epochs"])
    best_state, best_score, best_epoch, stale = copy.deepcopy(model.state_dict()), -1.0, 0, 0
    history = []
    positives = np.flatnonzero(y_train == 1)
    negatives = np.flatnonzero(y_train == 0)
    for epoch in range(epochs):
        model.train()
        if loss_kind == "smooth_ap":
            # сбалансированные батчи: фиксированная доля positives в каждом батче
            n_batches = max(int(config["ds_min_steps_per_epoch"]), len(train_rows) // batch_size)
            per_pos = max(2, int(batch_size * float(config["smooth_ap_positive_share"])))
            plan = [np.r_[rng.choice(positives, per_pos), rng.choice(negatives, batch_size - per_pos)]
                    for _ in range(n_batches)]
        else:
            # на маленьком прореженном train одной эпохи мало: минимум ds_min_steps_per_epoch шагов
            plan = []
            while len(plan) < max(int(config["ds_min_steps_per_epoch"]), math.ceil(len(train_rows) / batch_size)):
                order = rng.permutation(len(train_rows))
                plan.extend(order[i:i + batch_size] for i in range(0, len(order), batch_size))
        for idx in plan:
            x, t, m, c = batcher.batch(train_rows[idx])
            logits = model(torch.from_numpy(x).to(device), torch.from_numpy(t).to(device),
                           torch.from_numpy(m).to(device), torch.from_numpy(c).to(device))
            yb = torch.from_numpy(y_train[idx].astype(np.float32)).to(device)
            if loss_kind == "smooth_ap":
                loss = smooth_ap_loss(logits, yb, float(config["smooth_ap_tau"]))
                loss = loss + float(config["smooth_ap_bce_weight"]) * torch.nn.functional.binary_cross_entropy_with_logits(logits, yb)
            else:
                wb = torch.from_numpy(w_train[idx].astype(np.float32)).to(device)
                per = torch.nn.functional.binary_cross_entropy_with_logits(logits, yb, reduction="none")
                loss = (per * wb).sum() / wb.sum()
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimiser.step()
        raw_cal = _predict_ds(model, batcher, cal_rows, device, 4 * batch_size)
        score = float(average_precision_score(y_cal, raw_cal)) if 0 < y_cal.sum() < len(y_cal) else -float(np.mean((raw_cal - y_cal) ** 2))
        history.append(round(score, 5))
        if score > best_score + 1e-6:
            best_score, best_epoch, best_state, stale = score, epoch, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
            if stale >= int(config["ds_patience"]):
                break
    model.load_state_dict(best_state)
    return model, {"best_epoch": best_epoch, "epochs_run": len(history),
                   "calibration_score_by_epoch": history, "loss": loss_kind,
                   "early_stopping_on": "calibration_H1_pr_auc"}


# ---------------------------------------------------------------------------
# 4. Метрики (семантика 07) и бутстрэп
# ---------------------------------------------------------------------------
class EvalRows:
    def __init__(self, frame: Any, target: str, period_start: date):
        self.y = frame[target].to_numpy().astype(np.int8)
        entities = frame["d_object_key"].cast(str).to_numpy()
        _, self.entity = np.unique(entities, return_inverse=True)
        self.n_entities = int(self.entity.max()) + 1 if len(self.entity) else 0
        self.day = frame["d_cutoff_date"].cast(int).to_numpy().astype(np.int64)
        origin = (period_start - date(1970, 1, 1)).days
        self.week = np.maximum((self.day - origin) // 7, 0)
        self.n_weeks = int(self.week.max()) + 1 if len(self.week) else 1
        self.month = frame["d_cutoff_date"].dt.strftime("%Y-%m").to_numpy()


def fast_episodes(ev: EvalRows, p: np.ndarray, budget: int, cooldown_hours: int,
                  mask: np.ndarray | None = None) -> dict[str, Any]:
    """Эквивалент episode_metrics из 07 (threshold=None), проверяется самотестом."""
    rows = np.arange(len(p)) if mask is None else np.flatnonzero(mask)
    prob = _finite_probability(p)
    selected = np.zeros(len(p), dtype=bool)
    if len(rows):
        day = ev.day[rows]
        order = np.lexsort((np.arange(len(rows)), -prob[rows], day))  # стабильно, как argsort(kind=stable)
        sorted_rows = rows[order]
        sorted_day = day[order]
        new = np.r_[True, sorted_day[1:] != sorted_day[:-1]]
        start = np.maximum.accumulate(np.where(new, np.arange(len(sorted_rows)), 0))
        selected[sorted_rows[(np.arange(len(sorted_rows)) - start) < budget]] = True
    alerts = np.flatnonzero(selected)
    alerts = alerts[np.argsort(ev.day[alerts], kind="stable")]
    cooldown_days = cooldown_hours / 24.0
    accepted, last = [], {}
    for idx in alerts.tolist():
        e = int(ev.entity[idx])
        if e not in last or ev.day[idx] - last[e] >= cooldown_days:
            accepted.append(idx)
            last[e] = int(ev.day[idx])
    accepted = np.array(accepted, dtype=np.int64)
    hit = ev.y[accepted] == 1 if len(accepted) else np.zeros(0, bool)
    events = rows[ev.y[rows] == 1]
    event_hit = np.isin(events, accepted[hit]) if len(events) else np.zeros(0, bool)
    n_alerts, n_events = len(accepted), len(events)
    active_per_day = np.bincount(ev.day[rows] - ev.day[rows].min()) if len(rows) else np.zeros(1)
    alerts_per_day = np.bincount(ev.day[alerts] - ev.day[rows].min(), minlength=len(active_per_day)) if len(alerts) else np.zeros_like(active_per_day)
    days_with_rows = active_per_day > 0
    return {
        "alert_episodes": int(n_alerts), "proxy_episodes": int(n_events), "matched_episodes": int(hit.sum()),
        "precision": float(hit.sum() / n_alerts) if n_alerts else 0.0,
        "recall": float(event_hit.sum() / n_events) if n_events else 0.0,
        "alert_budget_per_day": budget, "cooldown_hours": cooldown_hours,
        "budget_saturation": float(np.mean(alerts_per_day[days_with_rows] / active_per_day[days_with_rows])) if days_with_rows.any() else None,
        "_alert_week": ev.week[accepted] if n_alerts else np.zeros(0, np.int64),
        "_alert_hit": hit, "_alert_entity": ev.entity[accepted] if n_alerts else np.zeros(0, np.int64),
        "_event_week": ev.week[events], "_event_hit": event_hit, "_event_entity": ev.entity[events],
    }


def public(d: dict[str, Any]) -> dict[str, Any]:
    return {k: v for k, v in d.items() if not k.startswith("_")}


class APPrep:
    def __init__(self, y: np.ndarray, p: np.ndarray):
        self.order = np.argsort(-p, kind="mergesort")
        sp = p[self.order]
        self.y = y[self.order].astype(float)
        self.ends = np.r_[np.flatnonzero(np.diff(sp) != 0), len(sp) - 1]

    def ap(self, w: np.ndarray) -> float:
        ww = w[self.order]
        tp = np.cumsum(ww * self.y)[self.ends]
        fp = np.cumsum(ww * (1 - self.y))[self.ends]
        if tp[-1] <= 0:
            return float("nan")
        precision = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1e-12), 0.0)
        return float(np.sum(np.diff(np.r_[0.0, tp / tp[-1]]) * precision))


def ci(values: np.ndarray) -> dict[str, Any]:
    v = values[np.isfinite(values)]
    if not len(v):
        return {"status": "not_evaluable", "mean_difference": None, "ci95_low": None, "ci95_high": None}
    return {"status": "ok", "mean_difference": float(v.mean()), "ci95_low": float(np.percentile(v, 2.5)),
            "ci95_high": float(np.percentile(v, 97.5)), "reps_finite": int(len(v))}


def point_metrics(ev: EvalRows, p: np.ndarray, threshold: float, min_pos: int,
                  mask: np.ndarray | None = None) -> dict[str, Any]:
    from sklearn.metrics import average_precision_score, brier_score_loss

    m = np.ones(len(p), bool) if mask is None else mask
    y, pp = ev.y[m], _finite_probability(p[m])
    positives = int(y.sum())
    evaluable = positives >= min_pos and positives < len(y)
    pr_auc = float(average_precision_score(y, pp)) if evaluable else None
    pred = pp >= threshold
    tp, fp = int(((y == 1) & pred).sum()), int(((y == 0) & pred).sum())
    return {
        "rows": int(len(y)), "positives": positives, "prevalence": float(y.mean()) if len(y) else None,
        "pr_auc": pr_auc, "pr_auc_status": "reported" if pr_auc is not None else f"not_evaluable_lt_{min_pos}_positives",
        "lift_over_prevalence": (pr_auc / float(y.mean())) if pr_auc is not None and y.mean() > 0 else None,
        "brier_score": float(brier_score_loss(y, pp)) if len(y) else None,
        "ece_10_bins": expected_calibration_error(y, pp) if len(y) else None,
        "threshold": float(threshold), "precision": tp / max(tp + fp, 1), "recall": tp / max(positives, 1),
        "false_alerts_per_1000_eligible_object_days": 1000 * fp / max(len(y), 1),
    }


def top_k_rows(ev: EvalRows, p: np.ndarray, k: int) -> dict[str, Any]:
    ep = fast_episodes(ev, p, k, 0)  # cooldown 0: чистый top-k в сутки
    return {"k": k, "alerts": ep["alert_episodes"], "precision": ep["precision"], "recall": ep["recall"]}


# ---------------------------------------------------------------------------
# 5. Основной прогон задачи
# ---------------------------------------------------------------------------
def run_task(task: str, config: dict[str, Any]) -> dict[str, Any]:
    started = time.perf_counter()
    spec = TASKS[task]
    target = spec["target"]
    out_dir = Path(os.environ.get("LDT_OUTPUT_DIR", config["output_dir"]))
    out_dir.mkdir(parents=True, exist_ok=True)
    registry = TARGET_REGISTRY[target]
    result: dict[str, Any] = {
        "status": None, "run_mode": config["run_mode"],
        "comparability": "non_comparable_smoke" if config["run_mode"] == "smoke" else "comparable_within_target",
        "task": task, "target_code": target, "target_semantics_ru": registry["meaning_ru"],
        "score_kind": registry["score_kind"], "evidence_level": registry["evidence_level"],
        "confirmed_incident_probability_reported": False,
        "config": config, "environment": device_info(),
        "evaluation_scope": "pre-2025 rolling folds (development validation), 2025 H2/2026 не используются",
    }
    try:
        device = _torch_device(config)
        result["environment"]["torch_device_used"] = str(device)
        object_day, channel, manifest, checks = load_bundle_checked(task, config)
    except (ContractError, DeviceError) as error:
        result.update(status="contract_failed" if isinstance(error, ContractError) else "gpu_required_not_available",
                      stop_reason=str(error))
        return finish(task, result, out_dir, None, started)
    result["input_contract"] = {**checks, "source_manifest_sha256": manifest.get("sha256")}
    keys = set(object_day["d_object_key"].cast(str).unique().to_list())
    od, elements = attach_elements(object_day, channel, spec["family_types"])
    del channel
    gc.collect()
    result["input_contract"]["max_set_size"] = int(elements["lengths"].max())
    result["input_contract"]["median_set_size"] = float(np.median(elements["lengths"]))
    result["input_contract"]["object_days_without_elements"] = int((elements["lengths"] == 0).sum())

    seed = int(config["random_seed"])
    min_pos = int(config["min_positives_for_pr_auc"])
    folds_out, store, predictions = {}, {}, []
    import polars as pl

    # Наличие F5 определяется только train/ранней частью H1; validation служит лишь оценке.
    f5_supported = all(
        int(rolling_split(od, target, fold)["train"][target].sum()) >= min_pos
        and int(rolling_split(od, target, fold)["calibration"].filter(
            pl.col("d_target_start_date") < fold.calibration_start + timedelta(
                days=(fold.calibration_end - fold.calibration_start).days // 2
            )
        )[target].sum()) >= min_pos
        for fold in ROLLING_FOLDS
    )
    for fold_index, fold in enumerate(ROLLING_FOLDS):
        log(f"=== {fold.name} ===")
        split = rolling_split(od, target, fold)
        train, cal, val = split["train"], split["calibration"], split["validation"]
        midpoint = fold.calibration_start + timedelta(days=(fold.calibration_end - fold.calibration_start).days // 2)
        cal_early = cal.filter(pl.col("d_target_start_date") < midpoint)
        cal_holdout = cal.filter(pl.col("d_target_start_date") >= midpoint)
        if cal_early.is_empty() or cal_holdout.is_empty():
            raise ContractError(f"{fold.name}: пустая половина calibration H1")
        y_early = cal_early[target].to_numpy().astype(np.int8)
        y_cal, ev = cal_holdout[target].to_numpy().astype(np.int8), EvalRows(val, target, fold.validation_start)
        sampled, weights = sample_training_rows(train, target, int(config["negative_to_positive_ratio"]), seed + fold_index)
        y_tr = sampled[target].to_numpy().astype(np.int8)
        pre = pl.concat([train, cal])
        fold_out: dict[str, Any] = {
            "rows": {"train": train.height, "train_sampled": sampled.height, "early_stopping": cal_early.height, "calibration": cal_holdout.height, "validation": val.height},
            "positives": {"train": int(train[target].sum()), "early_stopping": int(y_early.sum()), "calibration": int(y_cal.sum()), "validation": int(ev.y.sum())},
            "models": {}, "training": {},
        }
        raw: dict[str, tuple[np.ndarray, np.ndarray]] = {}  # model -> (calibration raw, validation raw)
        calibrate = {}
        prior = float(train[target].mean())
        raw["prevalence"] = (np.full(cal_holdout.height, prior), np.full(val.height, prior))
        calibrate["prevalence"] = False
        mapping, global_rate = composition_table(train, target, int(config["composition_min_rows"]))
        comp = lambda f: np.array([mapping.get(s, global_rate) for s in f["_sig"].to_list()])  # noqa: E731
        raw["composition_prevalence"] = (comp(cal_holdout), comp(val))
        calibrate["composition_prevalence"] = False
        fold_out["composition_signatures"] = {"fitted": len(mapping), "min_rows": int(config["composition_min_rows"]),
                                              "validation_rows_using_global_fallback": int(sum(s not in mapping for s in val["_sig"].to_list()))}
        raw["recurrence"] = (recurrence_score(cal_holdout, spec["days_since"]), recurrence_score(val, spec["days_since"]))
        raw["rule"] = (rule_score(cal_holdout, spec), rule_score(val, spec))
        logistic = LogisticObject(seed).fit(sampled, y_tr, weights)
        raw["logistic"] = (logistic.raw(cal_holdout), logistic.raw(val))
        artifacts = {"logistic": logistic, "prevalence": {"rate": prior},
                     "composition_prevalence": {"mapping": mapping, "global_rate": global_rate},
                     "recurrence": {"days_since_feature": spec["days_since"]},
                     "rule": {"days_since_feature": spec["days_since"],
                              "alarm_type_count_feature": spec["alarm_type_count"]}}
        log("простые модели готовы")

        # ---- Deep Sets ----
        train_elem_rows = sampled["_od"].to_numpy()
        elem_index = np.concatenate([np.arange(elements["offsets"][r], elements["offsets"][r + 1]) for r in train_elem_rows]) \
            if len(train_elem_rows) else np.zeros(0, np.int64)
        numeric = elements["numeric"][elem_index]
        median = np.nanmedian(numeric, axis=0)
        q25, q75 = np.nanquantile(numeric, 0.25, axis=0), np.nanquantile(numeric, 0.75, axis=0)
        scale = q75 - q25
        scale[~np.isfinite(scale) | (scale < 1e-6)] = 1.0
        median[~np.isfinite(median)] = 0.0
        robust = {"median": median, "scale": scale}
        vocab_types = sorted(set(elements["types"][elem_index].tolist()))
        vocab = {t: i + 1 for i, t in enumerate(vocab_types)}
        days = od[spec["days_since"]].cast(float).to_numpy().astype(float)
        context = np.stack([np.where(np.isfinite(days), np.log1p(np.maximum(days, 0)), 0.0),
                            (~np.isfinite(days)).astype(float)], axis=1)
        batcher = ElementBatcher(elements, robust, vocab, context, int(config["ds_max_set_size"]))
        early_rows, cal_rows, val_rows = cal_early["_od"].to_numpy(), cal_holdout["_od"].to_numpy(), val["_od"].to_numpy()
        ds_variants = ["deepsets"] + (["deepsets_transfer"] if task == "fire" else [])
        for variant in ds_variants:
            model = build_deepsets(len(ELEMENT_NUMERIC_FEATURES), len(vocab), context.shape[1], config)
            if variant == "deepsets_transfer":
                generic = train.filter(pl.col(GENERIC_ALARM_TARGET).is_not_null()) if GENERIC_ALARM_TARGET in train.columns else None
                pre_rows = generic["_od"].to_numpy()
                y_pre = generic[GENERIC_ALARM_TARGET].to_numpy().astype(np.int8)
                cal_g = cal_early.filter(pl.col(GENERIC_ALARM_TARGET).is_not_null())
                model, info_pre = train_deepsets(model, batcher, pre_rows, y_pre, np.ones(len(pre_rows)),
                                                 cal_g["_od"].to_numpy(), cal_g[GENERIC_ALARM_TARGET].to_numpy().astype(np.int8),
                                                 config, device, epochs=int(config["pretrain_epochs"]), seed=seed + 100 + fold_index)
                model, info = train_deepsets(model, batcher, train_elem_rows, y_tr, weights, early_rows, y_early, config, device,
                                             lr=float(config["ds_lr"]) * float(config["finetune_lr_factor"]), seed=seed + 200 + fold_index)
                info = {"pretrain_generic_alarm": info_pre, "finetune_fire": info}
            else:
                model, info = train_deepsets(model, batcher, train_elem_rows, y_tr, weights, early_rows, y_early, config, device,
                                             seed=seed + 300 + fold_index)
            raw[variant] = (_predict_ds(model, batcher, cal_rows, device, 1024), _predict_ds(model, batcher, val_rows, device, 1024))
            fold_out["training"][variant] = info
            artifacts[variant] = {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                                  "robust": robust, "vocab": vocab, "n_context": int(context.shape[1])}
            log(f"{variant}: лучшая эпоха {info.get('best_epoch', info.get('finetune_fire', {}).get('best_epoch'))}")
        # F5 обучается только при достаточной поддержке train/ранней части H1 каждого фолда.
        if task == "fire" and f5_supported:
            model = build_deepsets(len(ELEMENT_NUMERIC_FEATURES), len(vocab), context.shape[1], config)
            model, info = train_deepsets(model, batcher, train_elem_rows, y_tr, weights, early_rows, y_early, config, device,
                                         loss_kind="smooth_ap", seed=seed + 400 + fold_index)
            raw["deepsets_smooth_ap"] = (_predict_ds(model, batcher, cal_rows, device, 1024), _predict_ds(model, batcher, val_rows, device, 1024))
            fold_out["training"]["deepsets_smooth_ap"] = info
            artifacts["deepsets_smooth_ap"] = {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                                               "robust": robust, "vocab": vocab, "n_context": int(context.shape[1])}
        fold_out["truncated_sets"] = batcher.truncated_sets

        # ---- калибровка, порог, метрики ----
        probs: dict[str, np.ndarray] = {}
        calibrators: dict[str, Any] = {}
        thresholds: dict[str, float] = {}
        clean_mask = ~np.isfinite(val[spec["days_since"]].cast(float).to_numpy().astype(float)) | \
            (val[spec["days_since"]].cast(float).fill_null(1e9).to_numpy() >= int(config["clean_history_days"]))
        seen_objects = set(pre["d_object_key"].cast(str).unique().to_list())
        seen_mask = np.array([k in seen_objects for k in val["d_object_key"].cast(str).to_list()])
        slices = {"clean_history": clean_mask, "seen_object": seen_mask, "unseen_object": ~seen_mask}
        for name, (rc, rv) in raw.items():
            spec_platt = fit_platt(rc, y_cal) if calibrate.get(name, True) else {"kind": "identity"}
            calibrators[name] = spec_platt
            pc, pv = apply_platt(spec_platt, rc), apply_platt(spec_platt, rv)
            thr = threshold_for_precision(y_cal, pc, float(config["minimum_precision"]))
            probs[name], thresholds[name] = pv, thr
            entry = {"label": MODEL_LABELS[task].get(name, name), "point": point_metrics(ev, pv, thr, min_pos)}
            grid = {}
            for b in config["alert_budgets"]:
                for cd in config["cooldowns"]:
                    grid[f"budget_{b}_cooldown_{cd}h"] = public(fast_episodes(ev, pv, b, cd))
            entry["episodes"] = grid
            entry["top_k_per_day"] = [top_k_rows(ev, pv, k) for k in config["alert_budgets"]]
            entry["slices"] = {}
            for sname, smask in slices.items():
                if not smask.any():
                    entry["slices"][sname] = {"status": "not_evaluable", "reason": "empty_slice", "rows": 0}
                    continue
                s_ep = fast_episodes(ev, pv, int(config["alert_budget_per_day"]), int(config["cooldown_hours"]), mask=smask)
                entry["slices"][sname] = {**point_metrics(ev, pv, thr, min_pos, smask),
                                          "episode_recall": s_ep["recall"], "episode_precision": s_ep["precision"]}
            months = {}
            for month in sorted(set(ev.month.tolist())):
                mm = ev.month == month
                months[month] = {"rows": int(mm.sum()), "positives": int(ev.y[mm].sum()),
                                 "pr_auc": point_metrics(ev, pv, thr, min_pos, mm)["pr_auc"]}
            entry["monthly"] = months
            fold_out["models"][name] = entry
        fold_out["slices_definition"] = {
            "clean_history": f"{spec['days_since']} is null or >= {config['clean_history_days']}",
            "seen_object": "объект встречался в train+calibration фолда",
        }
        fold_out["slice_sizes"] = {k: {"rows": int(v.sum()), "positives": int(ev.y[v].sum())} for k, v in slices.items()}
        folds_out[fold.name] = fold_out
        store[fold.name] = {"ev": ev, "probs": probs, "thresholds": thresholds, "calibrators": calibrators, "slices": slices,
                            "val": val.select("d_object_key", "d_cutoff_date", "d_target_start_date",
                                              "d_target_end_date_exclusive", target),
                            "artifacts": artifacts}
        gc.collect()

    result["folds"] = folds_out
    support_ok = all(f["positives"]["validation"] >= min_pos for f in folds_out.values())
    result["support"] = {"min_validation_positives_per_fold": min_pos,
                         "validation_positives": {k: v["positives"]["validation"] for k, v in folds_out.items()},
                         "status": "ok" if support_ok else "not_evaluable_insufficient_positives"}
    if task == "fire":
        result["support"]["f5_smooth_ap"] = "trained" if f5_supported else "skipped_insufficient_train_or_early_h1_positives"
    select(task, result, store, config)
    result["status"] = "completed"
    return finish(task, result, out_dir, store, started, keys)


# ---------------------------------------------------------------------------
# 6. Бутстрэп, отбор, гипотезы
# ---------------------------------------------------------------------------
def select(task: str, result: dict[str, Any], store: dict[str, Any], config: dict[str, Any]) -> None:
    folds = result["folds"]
    names = list(next(iter(store.values()))["probs"])
    budget, cooldown = int(config["alert_budget_per_day"]), int(config["cooldown_hours"])
    key = f"budget_{budget}_cooldown_{cooldown}h"
    min_pos = int(config["min_positives_for_pr_auc"])

    def mean(name: str, getter) -> float | None:
        values = [getter(folds[f]["models"][name]) for f in folds]
        values = [v for v in values if v is not None]
        return float(np.mean(values)) if values else None

    pooled = {n: {
        "episode_recall": mean(n, lambda m: m["episodes"][key]["recall"]),
        "episode_precision": mean(n, lambda m: m["episodes"][key]["precision"]),
        "false_alerts_per_1000": mean(n, lambda m: m["point"]["false_alerts_per_1000_eligible_object_days"]),
        "pr_auc": mean(n, lambda m: m["point"]["pr_auc"]),
        "brier": mean(n, lambda m: m["point"]["brier_score"]),
        "budget_saturation": mean(n, lambda m: m["episodes"][key]["budget_saturation"]),
    } for n in names}
    result["pooled_mean_over_folds"] = pooled

    rng = np.random.default_rng(int(config["bootstrap_seed"]))
    reps = int(config["bootstrap_reps"])
    draws = {f: np.stack([np.bincount(rng.integers(0, s["ev"].n_weeks, s["ev"].n_weeks), minlength=s["ev"].n_weeks)
                          for _ in range(reps)]).astype(float) for f, s in store.items()}
    primaries = {f: {n: fast_episodes(s["ev"], s["probs"][n], budget, cooldown) for n in names} for f, s in store.items()}
    preps = {f: {n: APPrep(s["ev"].y, s["probs"][n]) for n in names} for f, s in store.items()}

    def paired(cand: str, ref: str, mask_name: str | None = None) -> dict[str, Any]:
        out, acc = {}, {"pr_auc": [], "episode_recall": [], "episode_precision": []}
        for f, s in store.items():
            ev, d = s["ev"], draws[f]
            mask = np.ones(len(ev.y)) if mask_name is None else s["slices"][mask_name].astype(float)
            entity_ok = None
            if mask_name is not None:
                entity_ok = np.zeros(ev.n_entities, bool)
                entity_ok[ev.entity[s["slices"][mask_name]]] = True
            per = {"pr_auc": np.full(reps, np.nan), "episode_recall": np.full(reps, np.nan),
                   "episode_precision": np.full(reps, np.nan)}
            pos_support = ev.y[mask > 0].sum()
            for r in range(reps):
                w = d[r][ev.week] * mask
                if pos_support >= min_pos:
                    per["pr_auc"][r] = preps[f][cand].ap(w) - preps[f][ref].ap(w)
                vals = []
                for item in (primaries[f][cand], primaries[f][ref]):
                    wa = d[r][item["_alert_week"]]
                    we = d[r][item["_event_week"]]
                    if entity_ok is not None:
                        wa = wa * entity_ok[item["_alert_entity"]]
                        we = we * entity_ok[item["_event_entity"]]
                    prec = (wa * item["_alert_hit"]).sum() / max(wa.sum(), 1e-12)
                    rec = (we * item["_event_hit"]).sum() / we.sum() if we.sum() > 0 else np.nan
                    vals.append((prec, rec))
                per["episode_precision"][r] = vals[0][0] - vals[1][0]
                per["episode_recall"][r] = vals[0][1] - vals[1][1]
            out[f] = {k: ci(v) for k, v in per.items()}
            for k, v in per.items():
                acc[k].append(v)
        with np.errstate(all="ignore"), __import__("warnings").catch_warnings():
            __import__("warnings").simplefilter("ignore", RuntimeWarning)
            out["mean_over_folds"] = {k: ci(np.nanmean(np.stack(v), axis=0)) for k, v in acc.items()}
        out["difference"] = f"{cand} - {ref}"
        return out

    simple = [n for n in SIMPLE_ORDER if n in names]
    rank = {n: i for i, n in enumerate(SIMPLE_ORDER)}
    best = max(simple, key=lambda n: (pooled[n]["episode_recall"] or -1))
    # при разнице < 0.01 и не худшей precision — предпочесть более простую модель
    for n in sorted(simple, key=lambda n: rank[n]):
        if (pooled[best]["episode_recall"] or 0) - (pooled[n]["episode_recall"] or 0) < 0.01 and \
                (pooled[n]["episode_precision"] or 0) >= (pooled[best]["episode_precision"] or 0) - 0.01:
            best = n
            break
    comparisons = {f"{n}_vs_{best}": paired(n, best) for n in names if n != best}
    comparisons["logistic_vs_recurrence"] = paired("logistic", "recurrence") if "logistic" in names else None
    if "deepsets_transfer" in names:
        comparisons["deepsets_transfer_vs_deepsets"] = paired("deepsets_transfer", "deepsets")
    if "deepsets_smooth_ap" in names:
        comparisons["deepsets_smooth_ap_vs_deepsets"] = paired("deepsets_smooth_ap", "deepsets")
    clean_ok = all(s["slices"]["clean_history"].any() for s in store.values())
    ranker = max([n for n in simple if n != "prevalence"], key=lambda n: pooled[n]["pr_auc"] or -1)
    if clean_ok:
        comparisons[f"clean_history:{ranker}_vs_prevalence"] = paired(ranker, "prevalence", "clean_history")
    result["paired_bootstrap"] = {"method": "парный бутстрэп блоками календарных недель validation; одинаковые блоки для всех моделей",
                                  "reps": reps, "seed": int(config["bootstrap_seed"]),
                                  "primary": f"episode recall @ {budget}/сутки, cooldown {cooldown} ч",
                                  "best_simple_reference": best, "comparisons": comparisons}

    support_ok = result["support"]["status"] == "ok"
    gates, passed = {}, []
    for n in [x for x in names if x in NEURAL]:
        c = comparisons[f"{n}_vs_{best}"]["mean_over_folds"]
        fa_limit = max(1.0, 0.05 * (pooled[best]["false_alerts_per_1000"] or 0))
        g = {
            "g1_primary_ci95_above_zero": bool(c["episode_recall"]["ci95_low"] is not None and c["episode_recall"]["ci95_low"] > 0),
            "g2_episode_precision_not_worse_by_0.01": bool((pooled[n]["episode_precision"] or 0) >= (pooled[best]["episode_precision"] or 0) - 0.01),
            "g3_false_alerts_not_higher": bool((pooled[n]["false_alerts_per_1000"] or 0) <= (pooled[best]["false_alerts_per_1000"] or 0) + fa_limit),
            "g4_support_ok": support_ok,
        }
        gates[n] = {"gates": g, "passed": all(g.values()), "failed": [k for k, v in g.items() if not v]}
        if all(g.values()):
            passed.append(n)
    if passed:
        chosen = max(passed, key=lambda n: (pooled[n]["episode_recall"], pooled[n]["episode_precision"]))
        reasons = ["NEURAL_SIGNIFICANT_AND_OPERATIONALLY_NONINFERIOR"]
    else:
        chosen = best
        reasons = ["PREFER_SIMPLE_BASELINE", "NEURAL_DID_NOT_PASS_ALL_GATES"] if gates else ["PREFER_SIMPLE_BASELINE"]
    saturation = pooled[best]["budget_saturation"]
    saturated = saturation is not None and saturation >= float(config["saturation_limit"])
    if saturated:
        # основной эпизодный бюджет покрывает почти все активные объекты — метрика не различает модели
        reasons.append("PRIMARY_BUDGET_SATURATED")
    if not support_ok:
        reasons = ["INSUFFICIENT_VALIDATION_POSITIVES"] + reasons
    status = ("not_evaluable_insufficient_positives" if not support_ok
              else "not_evaluable_primary_budget_saturated" if saturated else "selected_proxy_model")
    result["selection"] = {
        "selected_model": chosen, "selected_label": MODEL_LABELS[task].get(chosen, chosen),
        "decision_status": status,
        "backend_candidate": status == "selected_proxy_model",
        "neural_passed_symmetric_gates": bool(passed), "best_simple_model": best,
        "neural_gates": gates, "reason_codes": reasons,
        "primary_budget_saturation": saturation,
    }
    # Чувствительность: тот же отбор по более узкому бюджету (информативно при малом числе объектов)
    alt = int(config["sensitivity_budget"])
    alt_key = f"budget_{alt}_cooldown_{cooldown}h"
    alt_recall = {n: mean(n, lambda m: m["episodes"][alt_key]["recall"]) for n in names}
    alt_precision = {n: mean(n, lambda m: m["episodes"][alt_key]["precision"]) for n in names}
    result["selection"]["sensitivity_budget"] = {
        "budget": alt, "episode_recall": alt_recall, "episode_precision": alt_precision,
        "best_by_recall": max(names, key=lambda n: alt_recall[n] or -1),
        "note_ru": "Информативно; отбор по контракту идёт по основному бюджету.",
    }

    hyp = {}

    def decide(entry: dict[str, Any] | None, metric: str = "pr_auc") -> dict[str, Any]:
        if not support_ok:
            return {"decision": "not_evaluable", "reason": "INSUFFICIENT_VALIDATION_POSITIVES"}
        if entry is None:
            return {"decision": "not_evaluable", "reason": "COMPARISON_NOT_AVAILABLE"}
        c = entry["mean_over_folds"][metric]
        if c["ci95_low"] is None:
            return {"decision": "not_evaluable", "reason": "NO_FINITE_BOOTSTRAP"}
        if c["ci95_low"] > 0:
            return {"decision": "confirmed", "reason": "CI95_ABOVE_ZERO"}
        if c["ci95_high"] < 0:
            return {"decision": "rejected", "reason": "CI95_BELOW_ZERO"}
        return {"decision": "inconclusive", "reason": "CI95_INCLUDES_ZERO"}

    for n in [x for x in names if x in NEURAL]:
        hyp[f"{n}_beats_best_simple_on_primary"] = decide(comparisons[f"{n}_vs_{best}"], "episode_recall")
    hyp["logistic_beats_recurrence_pr_auc"] = decide(comparisons.get("logistic_vs_recurrence"))
    if "deepsets_transfer_vs_deepsets" in comparisons:
        hyp["transfer_from_generic_alarm_helps_pr_auc"] = decide(comparisons["deepsets_transfer_vs_deepsets"])
    if task == "fire":
        hyp["smooth_ap_surrogate_helps_pr_auc"] = decide(comparisons.get("deepsets_smooth_ap_vs_deepsets")) \
            if "deepsets_smooth_ap_vs_deepsets" in comparisons else {"decision": "not_evaluable", "reason": "SKIPPED_INSUFFICIENT_TRAIN_OR_EARLY_H1_POSITIVES"}
    key_clean = f"clean_history:{ranker}_vs_prevalence"
    hyp["strongest_simple_beats_prevalence_on_clean_history"] = (
        {**decide(comparisons[key_clean]), "model": ranker} if key_clean in comparisons
        else {"decision": "not_evaluable", "reason": "EMPTY_CLEAN_HISTORY_SLICE"})
    unseen = sum(result["folds"][f]["slice_sizes"]["unseen_object"]["positives"] for f in result["folds"])
    hyp["generalizes_to_unseen_objects"] = {"decision": "not_evaluable",
                                            "reason": "EMPTY_UNSEEN_OBJECT_SLICE" if unseen == 0 else "LT_20_POSITIVES" if unseen < min_pos else "NOT_TESTED_POINT_METRICS_ONLY"}
    result["hypotheses"] = hyp


def finish(task: str, result: dict[str, Any], out_dir: Path, store: dict[str, Any] | None,
           started: float, keys: set[str] | None = None) -> dict[str, Any]:
    import polars as pl

    spec = TASKS[task]
    res_name, pred_name, sum_name = spec["outputs"]
    result["runtime_seconds"] = round(time.perf_counter() - started, 1)
    result = _clean(result)
    if store and result.get("selection"):
        chosen = result["selection"]["selected_model"]
        parts = []
        for fold, s in store.items():
            parts.append(s["val"].with_columns(
                pl.Series("selected_score", s["probs"][chosen]),
                pl.lit(fold).alias("fold"),
                pl.lit(result["score_kind"]).alias("score_kind"),
                pl.lit(result["evidence_level"]).alias("evidence_level"),
            ))
        pred = pl.concat(parts, how="vertical_relaxed")
        allowed = ["d_object_key", "d_cutoff_date", "d_target_start_date", "d_target_end_date_exclusive",
                   spec["target"], "selected_score", "fold", "score_kind", "evidence_level"]
        assert pred.columns == allowed, pred.columns
        dup = pred.height - pred.select("d_object_key", "d_target_start_date").unique().height
        if dup:
            raise RuntimeError("В predictions повторяются object/window строки")
        pred.write_parquet(out_dir / pred_name, compression="zstd")
        result["predictions_file"] = {"rows": pred.height, "columns": allowed}
        if result["selection"]["backend_candidate"]:
            artifact = store[list(store)[-1]]["artifacts"].get(chosen)
            result["selection"]["artifact"] = save_artifact(task, chosen, artifact, store, out_dir, result)
    text = json.dumps(result, ensure_ascii=False, indent=2)
    if keys:
        leaked = {k for k in keys if len(k) >= 8} & set(re.findall(r"[A-Za-z0-9_\-]+", text))
        if leaked:
            raise RuntimeError(f"В результатах обнаружены ключи объектов ({len(leaked)}) — запись остановлена")
    (out_dir / res_name).write_text(text, encoding="utf-8")
    (out_dir / sum_name).write_text(render_summary(task, result), encoding="utf-8")
    log("Записано:", res_name, sum_name, pred_name if result.get("predictions_file") else "(predictions не созданы)")
    return result


def save_artifact(task: str, chosen: str, artifact: Any, store: dict[str, Any], out_dir: Path,
                  result: dict[str, Any]) -> dict[str, Any]:
    """Сохраняет модель последнего фолда (fold_2024) — без данных и ключей."""
    stem = TASKS[task]["artifact_stem"]
    fold = list(store)[-1]
    info = {"model": chosen, "trained_in_fold": fold, "threshold": store[fold]["thresholds"][chosen],
            "note_ru": "Артефакт development-фолда для воспроизведения; перед production нужен refit."}
    cal_spec = store[fold]["calibrators"][chosen]
    if cal_spec["kind"] == "platt":
        cal_model = cal_spec["model"]
        cal_payload = {"kind": "platt_logit", "coef": cal_model.coef_.ravel().tolist(),
                       "intercept": cal_model.intercept_.ravel().tolist(),
                       "classes": cal_model.classes_.tolist()}
    else:
        cal_payload = {"kind": "identity"}
    calibration_path = out_dir / f"{stem}_calibration.json"
    calibration_path.write_text(json.dumps({"calibrator": cal_payload,
        "threshold": float(info["threshold"]), "score_kind": TARGET_REGISTRY[TASKS[task]["target"]]["score_kind"]},
        ensure_ascii=False, indent=2), encoding="utf-8")
    info["calibration_file"] = calibration_path.name
    if chosen in NEURAL:
        import torch

        path = out_dir / f"{stem}.pt"
        torch.save({"state_dict": artifact["state_dict"], "robust_median": artifact["robust"]["median"].tolist(),
                    "robust_scale": artifact["robust"]["scale"].tolist(), "type_vocab": artifact["vocab"],
                    "element_features": ELEMENT_NUMERIC_FEATURES, "n_context": artifact["n_context"],
                    "context_days_since_feature": TASKS[task]["days_since"], "task": task,
                    "config": {k: result["config"][k] for k in result["config"] if k.startswith("ds_")}}, path)
    elif chosen == "logistic":
        import joblib

        path = out_dir / f"{stem}.joblib"
        joblib.dump(artifact.model, path)
        info["feature_order"] = OBJECT_FEATURES
        info["input_transform"] = "signed_log1p"
    else:
        path = out_dir / f"{stem}.json"
        path.write_text(json.dumps({"model": chosen, "kind": "deterministic_or_prevalence",
                                    "spec": artifact}, ensure_ascii=False, indent=2), encoding="utf-8")
    info["file"] = path.name
    return info


# ---------------------------------------------------------------------------
# 7. Сводка на русском
# ---------------------------------------------------------------------------
def _f(v: Any, d: int = 4) -> str:
    if v is None:
        return "—"
    if isinstance(v, int):
        return f"{v:,}".replace(",", " ")
    return f"{float(v):.{d}f}"


def _ci(c: dict[str, Any] | None) -> str:
    if not c or c.get("ci95_low") is None:
        return "—"
    return f"{c['mean_difference']:+.4f} [{c['ci95_low']:+.4f}; {c['ci95_high']:+.4f}]"


DECISION_RU = {"confirmed": "подтверждено", "rejected": "отклонено", "inconclusive": "не определено",
               "not_evaluable": "не оценивается"}


def render_summary(task: str, r: dict[str, Any]) -> str:
    spec = TASKS[task]
    L = [f"# {spec['title_ru']}", ""]
    L.append(f"Статус: **{r.get('status')}** · режим: **{r.get('run_mode')}** · `{r.get('comparability')}`")
    L.append("")
    if r.get("run_mode") == "smoke":
        L += ["> **SMOKE**: подвыборка объектов и эпох. Числа несравнимы.", ""]
    L += [f"Цель `{r['target_code']}` — {r['target_semantics_ru']}",
          f"Скор: `{r['score_kind']}`, уровень доказательности {r['evidence_level']}. "
          "Это weak-label proxy, **не подтверждённый инцидент**.", ""]
    if r.get("status") != "completed":
        L += ["## Остановка", "", f"Причина: {r.get('stop_reason')}", "", "Эксперименты не выполнялись."]
        return "\n".join(L) + "\n"
    sup = r["support"]
    L += ["## Поддержка", "", "| фолд | train | calibration | validation | positives validation |", "| --- | ---: | ---: | ---: | ---: |"]
    for f, v in r["folds"].items():
        L.append(f"| {f} | {_f(v['rows']['train'])} | {_f(v['rows']['calibration'])} | {_f(v['rows']['validation'])} | {_f(v['positives']['validation'])} |")
    L += ["", f"Статус поддержки: **{sup['status']}** (порог {sup['min_validation_positives_per_fold']} positives в каждом фолде)."]
    if sup["status"] != "ok":
        L += ["", "PR-AUC и её CI не публикуются как надёжные; гипотезы — `not_evaluable`; backend-кандидат не выбирается."]
    L.append("")
    sel = r["selection"]
    budget, cooldown = r["config"]["alert_budget_per_day"], r["config"]["cooldown_hours"]
    L += [f"## Модели (среднее по фолдам, эпизоды: {budget} тревог/сутки, cooldown {cooldown} ч)", "",
          "| модель | PR-AUC | episode recall | episode precision | ложных / 1000 | Brier | насыщение бюджета |",
          "| --- | ---: | ---: | ---: | ---: | ---: | ---: |"]
    for n, p in r["pooled_mean_over_folds"].items():
        label = MODEL_LABELS[task].get(n, n)
        L.append(f"| {label} `{n}` | {_f(p['pr_auc'])} | {_f(p['episode_recall'])} | {_f(p['episode_precision'])} "
                 f"| {_f(p['false_alerts_per_1000'], 1)} | {_f(p['brier'])} | {_f(p['budget_saturation'], 2)} |")
    L.append("")
    if (sel.get("primary_budget_saturation") or 0) >= r["config"]["saturation_limit"]:
        L += [f"> **Бюджет {budget} объектов/сутки насыщен** (в среднем тревожится {sel['primary_budget_saturation']:.0%} "
              "активных объектов в день): при ~десятках объектов основной эпизодный recall почти не различает модели. "
              f"См. чувствительность при бюджете {sel['sensitivity_budget']['budget']}.", ""]
    pb = r["paired_bootstrap"]
    L += [f"## Парные 95% CI против лучшей простой модели (`{pb['best_simple_reference']}`)", "",
          "| сравнение | Δ PR-AUC | Δ episode recall | Δ episode precision |", "| --- | ---: | ---: | ---: |"]
    for name, c in pb["comparisons"].items():
        if not c:
            continue
        m = c["mean_over_folds"]
        L.append(f"| {c['difference']} | {_ci(m['pr_auc'])} | {_ci(m['episode_recall'])} | {_ci(m['episode_precision'])} |")
    L.append("")
    L += ["## Срезы (основной бюджет)", "", "| фолд | срез | строк | positives | PR-AUC выбранной модели |", "| --- | --- | ---: | ---: | ---: |"]
    for f, v in r["folds"].items():
        for s, size in v["slice_sizes"].items():
            m = v["models"][sel["selected_model"]]["slices"].get(s, {})
            L.append(f"| {f} | {s} | {_f(size['rows'])} | {_f(size['positives'])} | {_f(m.get('pr_auc'))} |")
    L += ["", "Пустой unseen-срез означает «нет оценки», а не нулевую метрику.", ""]
    L += ["## Гипотезы", "", "| гипотеза | решение | причина |", "| --- | --- | --- |"]
    for k, h in r["hypotheses"].items():
        L.append(f"| `{k}` | **{DECISION_RU.get(h['decision'], h['decision'])}** | `{h['reason']}` |")
    L += ["", "## Отбор", "",
          f"Выбрано: **{sel['selected_label']} `{sel['selected_model']}`**, статус `{sel['decision_status']}`, "
          f"backend-кандидат: {'да' if sel['backend_candidate'] else 'нет'}.",
          f"Причины: {', '.join('`' + x + '`' for x in sel['reason_codes'])}.", ""]
    if sel["neural_gates"]:
        L += ["| нейросеть | прошла gates | не пройдены |", "| --- | --- | --- |"]
        for n, g in sel["neural_gates"].items():
            L.append(f"| {MODEL_LABELS[task].get(n, n)} `{n}` | {'да' if g['passed'] else 'нет'} | {', '.join(g['failed']) or '—'} |")
        L.append("")
    sb = sel["sensitivity_budget"]
    L += [f"Чувствительность (бюджет {sb['budget']}): лучший по episode recall — `{sb['best_by_recall']}`. Информативно.", "",
          "## Ограничения", "",
          "- Weak-label proxy по журналу; не подтверждённые проникновения/пожары.",
          "- Два pre-2025 фолда — development validation; 2025 H2 и 2026 не использовались.",
          "- H1 разделён по времени: ранняя остановка использует первую половину, Platt/порог — вторую.",
          "- Тихие object-days не достраивались; строки без метки не считаются отрицательными.", "",
          f"Время: {r.get('runtime_seconds')} с; устройство: {r['environment'].get('torch_device_used')}."]
    return "\n".join(L) + "\n"


def make_config(task: str, run_mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    if run_mode not in ("smoke", "full"):
        raise ValueError("run_mode: smoke или full")
    full = run_mode == "full"
    config = {
        "task": task, "run_mode": run_mode, "output_dir": "/kaggle/working", "random_seed": 20260922,
        "negative_to_positive_ratio": 20, "minimum_precision": 0.2,
        "alert_budget_per_day": 50, "cooldown_hours": 72, "sensitivity_budget": 5,
        "alert_budgets": [5, 10, 25, 50], "cooldowns": [24, 72, 168],
        "clean_history_days": 30, "min_positives_for_pr_auc": 20, "composition_min_rows": 100,
        "bootstrap_reps": 500 if full else 50, "bootstrap_seed": 42,
        "ds_hidden": 64, "ds_type_embedding": 8, "ds_dropout": 0.1, "ds_lr": 1e-3,
        "ds_batch_size": 256, "ds_min_steps_per_epoch": 50, "ds_epochs": 30 if full else 3, "ds_patience": 5, "ds_max_set_size": 1024,
        "pretrain_epochs": 10 if full else 2, "finetune_lr_factor": 0.3,
        "smooth_ap_tau": 0.25, "smooth_ap_positive_share": 0.0625, "smooth_ap_bce_weight": 0.05,
        "smoke_object_share": 3, "saturation_limit": 0.9,
    }
    config.update(overrides or {})
    return config


In [ ]:
"""Самопроверки 08/09 на синтетических данных (без панели)."""

from __future__ import annotations


from datetime import date, timedelta

import numpy as np


def _toy_object_days(n_objects: int = 12, n_days: int = 150, seed: int = 0):
    import polars as pl

    rng = np.random.default_rng(seed)
    rows = []
    for o in range(n_objects):
        for d in range(n_days):
            if rng.random() < 0.8:
                cut = date(2024, 7, 1) + timedelta(days=d)
                rows.append((f"obj_{o:02d}xxxxxxx", cut, cut + timedelta(days=2), cut + timedelta(days=3),
                             int(rng.random() < 0.08)))
    return pl.DataFrame(rows, schema=["d_object_key", "d_cutoff_date", "d_target_start_date",
                                      "d_target_end_date_exclusive", ACCESS_TARGET], orient="row")


def run_self_tests() -> dict[str, bool]:
    import torch
    from sklearn.metrics import average_precision_score

    report: dict[str, bool] = {}
    frame = _toy_object_days()
    ev = EvalRows(frame, ACCESS_TARGET, date(2024, 7, 1))
    rng = np.random.default_rng(1)
    p = np.round(rng.random(frame.height) * 0.6 + 0.4 * ev.y * rng.random(frame.height), 2)  # есть равные скоры

    # 1. Быстрые эпизоды = episode_metrics из 07 (threshold=None).
    for budget, cooldown in ((1, 24), (3, 72), (5, 168), (50, 72)):
        mine = fast_episodes(ev, p, budget, cooldown)
        ref = episode_metrics(frame, ACCESS_TARGET, p, budget, cooldown, None, "d_object_key")
        for key in ("alert_episodes", "proxy_episodes", "matched_episodes"):
            assert mine[key] == ref[key], (budget, cooldown, key, mine[key], ref[key])
    report["episodes_match_runtime_07"] = True

    # 2. Взвешенный PR-AUC = sklearn.
    w = rng.integers(0, 4, len(p)).astype(float)
    assert abs(APPrep(ev.y, p).ap(w) - average_precision_score(ev.y, p, sample_weight=w)) < 1e-9
    report["weighted_pr_auc_matches_sklearn"] = True

    # 3. Deep Sets: инвариантность к перестановке элементов и к паддингу.
    torch.manual_seed(0)
    config = make_config("access", "smoke")
    model = build_deepsets(4, 5, 2, config).eval()
    x = torch.randn(3, 6, 4)
    t = torch.randint(1, 6, (3, 6))
    mask = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1], [1, 0, 0, 0, 0, 0]], dtype=torch.float32)
    ctx = torch.randn(3, 2)
    with torch.no_grad():
        base = model(x, t, mask, ctx)
        perm = torch.tensor([2, 0, 1, 3, 4, 5])
        x2, t2 = x.clone(), t.clone()
        x2[0], t2[0] = x[0, perm], t[0, perm]
        assert torch.allclose(model(x2, t2, mask, ctx), base, atol=1e-6)
        x3 = x.clone()
        x3[0, 3:] = 1e6  # мусор в паддинге не должен влиять
        assert torch.allclose(model(x3, t, mask, ctx), base, atol=1e-6)
    report["deepsets_permutation_invariant"] = True
    report["deepsets_padding_masked"] = True

    # 4. Неизвестный тип получает зарезервированный индекс 0.
    elements = {"numeric": np.zeros((3, len(ELEMENT_NUMERIC_FEATURES))), "types": np.array(["A", "B", "Новый"]),
                "offsets": np.array([0, 3]), "lengths": np.array([3])}
    batcher = ElementBatcher(elements, {"median": np.zeros(len(ELEMENT_NUMERIC_FEATURES)),
                                        "scale": np.ones(len(ELEMENT_NUMERIC_FEATURES))},
                             {"A": 1, "B": 2}, np.zeros((1, 2)), 10)
    assert batcher.t.tolist() == [1, 2, 0]
    report["unknown_type_reserved_index"] = True

    # 5. Smooth-AP: лучше ранжированные positives дают меньший loss; loss дифференцируем.
    y = torch.tensor([1.0, 0, 0, 1, 0, 0])
    good = torch.tensor([3.0, -1, -2, 2.5, -1.5, -3], requires_grad=True)
    bad = torch.tensor([-3.0, 1, 2, -2.5, 1.5, 3])
    loss_good = smooth_ap_loss(good, y, 0.25)
    assert loss_good < smooth_ap_loss(bad, y, 0.25)
    loss_good.backward()
    assert good.grad is not None and torch.isfinite(good.grad).all()
    exact = average_precision_score(y.numpy(), good.detach().numpy())
    assert abs((1 - loss_good.detach().item()) - exact) < 0.05
    report["smooth_ap_orders_and_differentiable"] = True
    return report


## Самопроверки на синтетических данных

Быстрые эпизоды совпадают с `episode_metrics` из 07; взвешенный PR-AUC — со `sklearn`; Deep Sets инвариантен к перестановке элементов и к мусору в паддинге; неизвестный тип получает индекс 0; Smooth-AP дифференцируем и монотонен.

In [ ]:
SELF_TESTS = run_self_tests()
for name, ok in SELF_TESTS.items():
    print("OK  " if ok else "FAIL", name)
assert all(SELF_TESTS.values()), "Самопроверки не пройдены"

## Основной прогон

In [ ]:
RUN_CONFIG = make_config(TASK, CONFIG["run_mode"], {k: v for k, v in CONFIG.items() if k != "run_mode"})
RESULT = run_task(TASK, RUN_CONFIG)
print()
print("Статус:", RESULT["status"])
if RESULT.get("stop_reason"):
    print("Причина остановки:", RESULT["stop_reason"])
if RESULT.get("selection"):
    s = RESULT["selection"]
    print("Выбрано:", s["selected_label"], s["selected_model"], "|", s["decision_status"], "|", ", ".join(s["reason_codes"]))

## Сводка

In [ ]:
import os
from pathlib import Path
from IPython.display import Markdown, display
OUT = Path(os.environ.get("LDT_OUTPUT_DIR", RUN_CONFIG["output_dir"]))
display(Markdown((OUT / "summary_access_missing_target_ru.md").read_text(encoding="utf-8")))

## Графики

Слева — PR-AUC по моделям (среднее по фолдам; пустой столбец = `not_evaluable`). Справа — эпизодный recall в зависимости от бюджета тревог в сутки для трёх моделей: prevalence, лучшей простой по PR-AUC и лучшей нейросети.

In [ ]:
import matplotlib.pyplot as plt
if RESULT.get("status") == "completed":
    TEXT, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3de"
    SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # слоты 1–3 валидированной палитры (попарно различимы, в т.ч. при CVD)
    pooled = RESULT["pooled_mean_over_folds"]
    names = list(pooled)
    labels = [f"{MODEL_LABELS[TASK].get(n, n)}" for n in names]
    values = [pooled[n]["pr_auc"] for n in names]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), facecolor="#fcfcfb")
    ax = axes[0]
    if all(v is None for v in values):
        ax.text(0.5, 0.5, "PR-AUC не оценивается:\nменьше 20 positives в validation фолда", ha="center", va="center",
                color=MUTED, fontsize=11, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        top = max(v for v in values if v is not None)
        ax.barh(labels, [v or 0 for v in values], color=SERIES[0], height=0.55)
        for i, v in enumerate(values):
            ax.text((v or 0) + top * 0.02, i, "не оценивается" if v is None else f"{v:.3f}", va="center", color=TEXT, fontsize=9)
        ax.set_xlim(0, top * 1.2); ax.invert_yaxis()
    ax.set_title("PR-AUC (среднее по фолдам)", color=TEXT, loc="left")
    ax.set_xlabel("PR-AUC", color=MUTED)
    ax = axes[1]
    folds = RESULT["folds"]
    budgets = RESULT["config"]["alert_budgets"]
    cooldown = RESULT["config"]["cooldown_hours"]
    simple = [n for n in names if n not in NEURAL and n != "prevalence"]
    ranker = max(simple, key=lambda n: pooled[n]["pr_auc"] or -1)
    neural = [n for n in names if n in NEURAL]
    chosen = ["prevalence", ranker] + ([max(neural, key=lambda n: pooled[n]["pr_auc"] or -1)] if neural else [])
    for color, n in zip(SERIES, chosen):
        ys = [sum(folds[f]["models"][n]["episodes"][f"budget_{b}_cooldown_{cooldown}h"]["recall"] for f in folds) / len(folds) for b in budgets]
        ax.plot(budgets, ys, color=color, linewidth=2, marker="o", markersize=8, label=MODEL_LABELS[TASK].get(n, n))
    ax.set_xscale("log"); ax.set_xticks(budgets); ax.set_xticklabels([str(b) for b in budgets])
    ax.set_title(f"Episode recall от бюджета (cooldown {cooldown} ч)", color=TEXT, loc="left")
    ax.set_xlabel("тревог объектов в сутки", color=MUTED); ax.set_ylabel("episode recall", color=MUTED)
    ax.legend(frameon=False)
    for a in axes:
        a.set_facecolor("#fcfcfb"); a.set_axisbelow(True); a.grid(axis="x" if a is axes[0] else "y", color=GRID, linewidth=0.8)
        for side in ("top", "right"): a.spines[side].set_visible(False)
        a.tick_params(colors=MUTED)
    fig.tight_layout(); plt.show()   # график не сохраняется в output
else:
    print("Графиков нет: прогон остановлен.")

In [ ]:
print("Файлы для передачи команде:")
for name in ("results_access_missing_target.json", "predictions_access_missing_target.parquet", "summary_access_missing_target_ru.md"):
    p = OUT / name
    print("-", name, f"({p.stat().st_size / 1024:.0f} КБ)" if p.exists() else "— НЕ СОЗДАН")
for p in sorted(OUT.glob("model_access_selected.*")):
    print("-", p.name, "(артефакт выбранной модели)")
print("В predictions — только псевдонимный ключ, даты, цель, выбранный скор, фолд, score_kind, evidence_level.")